# COMP0178 - Lesson 1b: SQL

- Explained bullet-point notes for **all 75 pages** of *1B. SQL*, Week 1, slides adapted from John Dowell.
- Source: [Lecture_1b.pdf](Lectures/Lecture_1b.pdf).
- Page references use PDF page order.
- SQL snippets are study examples in Markdown, not automatically executed Python cells.
- The slides mix general SQL with MySQL examples. Corrections and dialect qualifications are identified explicitly.
- The sample names below are standardized to **Branch, Staff, Client, Viewing, Property**. Where address components are queried, assume those columns exist; the slide schema abbreviates address fields and varies singular/plural names.

## Runnable practice: how to use this notebook

- Read a topic, attempt the exercise, run your answer, then open its solution to compare reasoning.
- Choose a **Python 3 Jupyter kernel** and run the setup cell below once per kernel session.
- `%%sql` cells run actual SQL against a disposable **in-memory SQLite** database. No package installation or database server is required beyond the Jupyter Python environment.
- This notebook registers its own small SQL magic; it does not require JupySQL. The SQL practice uses syntax shared by SQLite and MySQL where possible.
- MySQL-only commands such as CREATE DATABASE, USE, ENGINE, GRANT, ANY/ALL, and some ALTER forms remain labelled lecture examples; do not run those against SQLite.
- Each SQL cell runs **one statement**. Python cells run setup/checks; Markdown cells hold notes, questions, and solutions.
- Run an answer cell immediately before its check. Comments-only starter cells prompt you to write a query without revealing the answer.
- `lab.reset()` restores the original dataset; the database is not saved when the kernel closes. Each notebook should run its own setup.
- Checks compare results on this dataset, including duplicates. They do not prove correctness for every possible input.
- To practice MySQL specifically later, use a separate MySQL/JupySQL connection rather than mixing it with this SQLite session.

In [ ]:
from pathlib import Path
import sys

# Find the helper whether Jupyter starts in COMP0178 or on the Desktop.
practice_folder = next((p for p in [Path.cwd(), Path.cwd() / 'COMP0178',
    Path('/Users/peidi/Desktop/COMP0178')] if (p / 'sql_practice.py').exists()), None)
if practice_folder is None:
    raise FileNotFoundError('Keep sql_practice.py beside these notebooks')
sys.path.insert(0, str(practice_folder))
from sql_practice import Practice
lab = Practice()
lab.register(get_ipython())

## 1. What SQL is (pages 1-4)

- **SQL = Structured Query Language**, a standardized language for using relational databases.
- It is **declarative/non-procedural**: specify the result you want rather than programming every search step. The DBMS chooses an execution plan.
- It provides **DDL** for defining structure and **DML** for reading/changing records.
- SQL keywords are case-insensitive; uppercase keywords are a readability convention.
- Use single quotes for string literals and semicolons to end statements.
- Formatting whitespace between tokens generally does not change meaning; whitespace inside quoted strings does.
- The slides say stored data is case-sensitive. Qualification: string comparison depends on data type and **collation**; MySQL comparisons are often case-insensitive under a case-insensitive collation.
- SQL dialects differ. This module focuses on **MySQL**, which can be used through **phpMyAdmin**.
- phpMyAdmin is a management interface, not a DBMS or the SQL language itself.
- The lecture is a reference covering creation, updates, filtering, aggregation, subqueries, joins, and advanced features.

## 2. Create a database and define tables (pages 5-7)

- `CREATE DATABASE` creates a database; `USE` selects it for subsequent statements.
- `CREATE TABLE` specifies columns, types, and optional constraints.
- Define referenced parent tables before dependent child tables.

```sql
CREATE DATABASE EstateAgent;
USE EstateAgent;

CREATE TABLE Branch (
    branchNo VARCHAR(4) PRIMARY KEY,
    street VARCHAR(100),
    city VARCHAR(50),
    postcode VARCHAR(10)
) ENGINE=InnoDB;

CREATE TABLE Staff (
    staffNo VARCHAR(4) PRIMARY KEY,
    fName VARCHAR(20) NOT NULL,
    lName VARCHAR(20) NOT NULL,
    position VARCHAR(20),
    sex CHAR(1),
    DOB DATE,
    salary DECIMAL(10,2),
    branchNo VARCHAR(4),
    CONSTRAINT fk_staff_branch
        FOREIGN KEY (branchNo) REFERENCES Branch(branchNo)
) ENGINE=InnoDB;
```

- This is a corrected supplementary version of the slide's Staff definition.
- The slide uses `DOB VARCHAR(15)`; `DATE` models a date more directly.
- The slide's `salary DECIMAL(5,2)` allows only three digits before the decimal, so it cannot hold the later salary example 21000. `DECIMAL(10,2)` avoids that mismatch.

## 3. Data types and defaults (pages 8-9)

- **CHAR(n):** fixed-length character data; **VARCHAR(n):** variable-length strings up to n characters.
- **INT:** integers; **DECIMAL(p,s):** exact numeric values with p total digits and s fractional digits.
- **FLOAT:** approximate numeric values; exact decimal amounts are usually more suitable for prices/salaries.
- **DATE, TIME, TIMESTAMP:** dates, times, and date/time-related values, with details depending on dialect.
- The slides also introduce bit strings, intervals, and large character/binary objects; type names/support vary by product.
- SQL conditions use **TRUE, FALSE, UNKNOWN**, with UNKNOWN arising from NULL comparisons. This is three-valued logic, not three ordinary stored Boolean values.
- MySQL BOOLEAN is an alias of a small integer type; it does not by itself enforce that stored values are only 0 and 1.
- A **DEFAULT** supplies a value when an INSERT omits that column.
- Example: `status VARCHAR(20) DEFAULT 'Open'`.
- The slide's current-date-plus-21-days expression illustrates a computed date default; exact expression-default syntax and support depend on server/version. The `DAYS` spelling is not a MySQL-ready example.

## 4. Integrity constraints (pages 10-14)

- Constraints declare rules that the database should enforce, instead of relying entirely on application code.
- **Required data:** `NOT NULL` prevents a missing value; it does not prevent an empty string.
- **Entity integrity:** `PRIMARY KEY` identifies a row uniquely and implies non-null values.
- **UNIQUE:** prevents duplicate key values; it is not equivalent to NOT NULL. MySQL unique indexes can permit multiple NULL values.
- A composite primary key is declared as `PRIMARY KEY (column1, column2)`.
- **Referential integrity:** `FOREIGN KEY (branchNo) REFERENCES Branch(branchNo)` requires each non-null reference to match a parent key.
- **Domain constraints:** types and CHECK rules restrict allowable values.
- **General constraints:** additional rules about the data, sometimes involving multiple fields.
- Constraints may be named using `CONSTRAINT name ...`, making them easier to identify/manage.
- Foreign-key actions specify what happens to dependent rows when the parent key is updated or the parent is deleted.
- **CASCADE:** propagate the key update or delete dependent rows.
- **SET NULL:** retain the child row but remove the reference; the child column must allow NULL.
- **NO ACTION/RESTRICT:** with InnoDB, reject a parent change that would leave invalid references.
- **SET DEFAULT:** a general SQL option shown in the slides, but rejected by MySQL InnoDB; do not use it as an InnoDB feature.
- Explanatory choice: deleting a Branch should not necessarily delete all its Staff; select actions according to the intended business rule.

## 5. Engines, ALTER, and DROP (pages 15-17)

- MySQL has storage engines; `ENGINE=InnoDB` selects InnoDB for a table.
- The lecture identifies InnoDB as the default and associates it with foreign-key enforcement and ACID transactions; MyISAM does not provide those same guarantees.
- The slide's blanket claim that MyISAM is better for searching is not a universal performance rule: workload and configuration matter.
- `ALTER TABLE` changes an existing table's definition.
- `DROP TABLE` removes the table definition and its contents.

```sql
ALTER TABLE Staff ADD COLUMN email VARCHAR(255);
ALTER TABLE Staff DROP COLUMN email;
ALTER TABLE Staff ALTER COLUMN position SET DEFAULT 'Assistant';
ALTER TABLE Staff DROP FOREIGN KEY fk_staff_branch;
DROP TABLE Staff;
```

- These are alternatives illustrating syntax, not a setup script to run sequentially before the later queries.
- MySQL has constraint-specific DROP syntax; the generic slide template `DROP CONSTRAINT` is not interchangeable with every constraint-specific command.

## 6. INSERT, UPDATE, DELETE, and importing data (pages 18-23)

- **INSERT** adds records; **SELECT** retrieves them; **UPDATE** changes values; **DELETE** removes records.
- Import data with phpMyAdmin's Import tools or an appropriately configured `LOAD DATA LOCAL INFILE` statement.
- File format, delimiters, permissions, and local-infile settings affect importing.
- Name INSERT columns explicitly so the value-to-column mapping is clear.
- Use ISO-style date strings for DATE columns, not the slide's ambiguous `'12-09-90'`.

```sql
INSERT INTO Branch (branchNo, street, city)
VALUES ('B003', '163 Main St', 'Glasgow');

INSERT INTO Staff (staffNo, fName, lName, position, sex, DOB, salary, branchNo)
VALUES ('SG17', 'Joe', 'Brown', 'Manager', 'M', '1990-09-12', 21000, 'B003');

UPDATE Staff SET salary = salary * 1.03;
UPDATE Staff SET salary = salary * 0.95 WHERE position = 'Manager';

DELETE FROM Staff WHERE staffNo = 'SG17';
```

- The first UPDATE affects all staff; the second affects only managers.
- `DELETE FROM Staff;` removes **every row**, leaving the table structure.
- `DROP TABLE Staff;` removes the table itself.
- Before an UPDATE or DELETE, understand the rows selected by its WHERE condition.

## 7. Anatomy of SELECT and the example schema (pages 24-29)

- The slides use real-estate data: Staff, Branch, Client/Clients, Viewing, and Property.
- Relationships include Staff→Branch, Property→Staff, and Viewing→Client/Property.
- SELECT chooses output columns; FROM supplies source tables; WHERE filters rows; GROUP BY forms groups; HAVING filters groups; ORDER BY sorts output.

```sql
SELECT staffNo, fName, lName, salary
FROM Staff;

SELECT * FROM Staff;

SELECT DISTINCT propertyNo FROM Viewing;

SELECT staffNo, salary / 12 AS monthlySal FROM Staff;
```

- `*` selects all columns.
- **DISTINCT** removes duplicate output rows; it does not change the stored records.
- `AS` gives an output expression a readable alias.
- Calculating `salary/12` assumes salary is annual; it does not update salary in the database.
- Without ORDER BY, output order is not guaranteed.
- A useful logical reading order is FROM/JOIN → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT. This explains results, not the physical execution plan.

### Practice B1

- **Task:** Return staffNo, fName, and salary for every staff member, ordered by staffNo.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 5 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([('S001', 'Ada', 30000), ('S002', 'Ben', 18000), ('S003', 'Cara', 24000), ('S004', 'Dan', None), ('S005', 'Eve', 20000)], ordered=True)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT staffNo, fName, salary FROM Staff ORDER BY staffNo;
```

- SELECT chooses columns; ORDER BY makes output deterministic.

</details>

## 8. WHERE filters: comparisons, ranges, membership, and patterns (pages 30-34)

- Comparisons include `=`, `<>`, `<`, `>`, `<=`, `>=`.
- Combine conditions with **AND, OR, NOT**. Use parentheses when mixing them to make the intended logic explicit.
- **BETWEEN includes both endpoints**.
- **IN** checks membership in a list; NOT IN checks non-membership, with NULL qualifications discussed below.
- **LIKE** uses `%` for zero or more characters and `_` for exactly one character.
- LIKE patterns are **not full regular expressions**, despite the slide's description as regex.

```sql
SELECT staffNo, salary FROM Staff WHERE salary > 10000;
SELECT * FROM Branch WHERE city = 'London' OR city = 'Glasgow';
SELECT * FROM Staff WHERE salary BETWEEN 20000 AND 30000;
SELECT * FROM Staff WHERE position IN ('Manager', 'Supervisor');
SELECT propertyNo FROM Property WHERE street LIKE '%awr%';
```

- `LIKE 'A%'` finds strings starting with A; `LIKE '_A%'` finds strings whose second character is A.
- The Property query assumes a street column, although the earlier abbreviated schema labels the location simply address.
- String comparisons and LIKE matching can depend on collation.

### Practice B2

- **Task:** Return staffNo and salary for Managers or Supervisors earning at least 24000, ordered by salary descending.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 2 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([('S001', 30000), ('S003', 24000)], ordered=True)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT staffNo, salary FROM Staff WHERE position IN ('Manager','Supervisor') AND salary >= 24000 ORDER BY salary DESC;
```

- Both the role condition and salary condition must hold. NULL salary does not satisfy the comparison.

</details>

### Practice B3

- **Task:** Return propertyNo for properties with Park in the street name, ordered by propertyNo.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 2 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([('P002',), ('P004',)], ordered=True)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT propertyNo FROM Property WHERE street LIKE '%Park%' ORDER BY propertyNo;
```

- % matches any surrounding sequence; LIKE is not full regex.

</details>

## 9. NULL and sorting (pages 35-37)

- **NULL** means missing/unknown information, not numeric zero or an empty string.
- Use `IS NULL` or `IS NOT NULL`, not `= NULL` or `<> NULL`.
- WHERE retains rows only when its condition is TRUE; UNKNOWN is filtered out.
- A list/subquery containing NULL can make NOT IN evaluate UNKNOWN rather than TRUE for an unmatched value. A suitable NOT EXISTS query often avoids this trap.
- ORDER BY defaults to ASC; DESC reverses the order.
- Multiple ordering expressions are applied in sequence.

```sql
SELECT clientNo, viewDate FROM Viewing WHERE comment IS NULL;
SELECT staffNo, salary FROM Staff ORDER BY salary DESC, staffNo ASC;
SELECT propertyNo, type, rooms, rent FROM Property ORDER BY type ASC, rent DESC;
```

- The last query first groups the ordering by type, then puts higher rents first within each type.
- A secondary unique key can make tie ordering deterministic.

### Practice B4

- **Task:** Return staffNo and fName for staff whose salary is missing.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 1 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([('S004', 'Dan')], ordered=False)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT staffNo, fName FROM Staff WHERE salary IS NULL;
```

- IS NULL is the missing-value test; = NULL would not select the row.

</details>

## 10. Aggregates and NULL (pages 38-43)

- Aggregates calculate a result over a set of rows: **COUNT, SUM, AVG, MIN, MAX**.
- `COUNT(*)` counts rows, including rows with NULL columns.
- `COUNT(column)` counts non-null values in that column.
- `COUNT(DISTINCT column)` counts distinct non-null values.
- SUM/AVG operate on numeric values; MIN/MAX can also compare nonnumeric values.
- Aggregates generally ignore NULL inputs. COUNT returns 0 for no qualifying values; SUM/AVG/MIN/MAX return NULL when no non-null values qualify.
- Example salaries `[10000,20000,NULL]`: COUNT(*)=3, COUNT(salary)=2, SUM=30000, AVG=15000. The NULL is not treated as zero in AVG.

```sql
SELECT COUNT(*) AS managers, SUM(salary) AS totalSalary
FROM Staff WHERE position = 'Manager';

SELECT MIN(salary), MAX(salary), AVG(salary) FROM Staff;

SELECT COUNT(DISTINCT propertyNo) AS propertiesViewed
FROM Viewing
WHERE viewDate >= '2012-05-01' AND viewDate < '2012-06-01';
```

- The half-open date range includes all of May, including late times on May 31 if the field is a timestamp.
- Do not mix an ungrouped staffNo with SUM(salary): one total does not identify one particular staff record.
- The slide says aggregates can only be used in SELECT/HAVING; ORDER BY can also use aggregate expressions. The key introductory restriction is that a group's aggregate cannot be placed directly in that same query block's WHERE.

### Practice B5

- **Task:** Return one row with total staff count, known salary count, and average salary in that order.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 1 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([(5, 4, 23000.0)], ordered=False)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT COUNT(*), COUNT(salary), AVG(salary) FROM Staff;
```

- Five staff exist, but only four salaries are known; 92000/4 = 23000.

</details>

## 11. GROUP BY and HAVING (pages 44-47)

- GROUP BY partitions rows by matching grouping values; aggregations then run separately in each group.
- **WHERE filters individual rows before grouping**; **HAVING filters groups after aggregation**.
- As a portable beginner rule, put every selected nonaggregated column in GROUP BY. MySQL can also recognize some functional dependencies, but do not depend on arbitrary ungrouped values.

```sql
SELECT branchNo, COUNT(*) AS staffCount, SUM(salary) AS totalSalary
FROM Staff
WHERE salary >= 10000
GROUP BY branchNo
HAVING COUNT(*) > 1
ORDER BY branchNo
LIMIT 30 OFFSET 0;
```

- First retain staff earning at least 10000; then count those retained staff per branch; then keep branches with more than one retained staff member.
- Removing WHERE changes the meaning to counting all staff.
- MySQL's `LIMIT 0, 30` in the slide means offset 0, at most 30 output rows, not “rows 0 to 30 inclusive.”
- Example: two retained B003 salaries 12000 and 24000 produce staffCount 2 and totalSalary 36000.

### Practice B6

- **Task:** Return branchNo, staff count, and total salary for branches with at least three staff.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 1 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([('B002', 3, 44000)], ordered=False)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT branchNo, COUNT(*), SUM(salary) FROM Staff GROUP BY branchNo HAVING COUNT(*) >= 3;
```

- HAVING filters groups; B002 contains three staff and two non-null salaries.

</details>

## 12. Scalar subqueries and IN (pages 48-51)

- A **subquery** is a SELECT nested inside another statement.
- A scalar subquery supplies one column and at most one row. Multiple rows cause an error; zero rows supply NULL.
- A single-column multirow result can be used with IN; a more general subquery can produce a full table.
- The slides' “0-D/1-D/2-D” terminology describes result shapes informally.

```sql
SELECT staffNo, fName, salary FROM Staff
WHERE salary > (SELECT AVG(salary) FROM Staff);

SELECT staffNo, fName FROM Staff
WHERE branchNo IN (
    SELECT branchNo FROM Branch WHERE street = '163 Main St'
);

SELECT propertyNo FROM Property
WHERE staffNo IN (
    SELECT staffNo FROM Staff
    WHERE branchNo IN (
        SELECT branchNo FROM Branch WHERE street = '163 Main St'
    )
);
```

- First compute the average in its own query block, then compare each staff salary to it.
- The slide uses `=` for the branch-address subquery. That assumes at most one matching branch; IN handles multiple matches.
- The nested Property query follows the relationship chain **address → branch → staff → properties**.

### Practice B7

- **Task:** Return staffNo and salary for staff earning more than the overall average, ordered by salary descending.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 2 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([('S001', 30000), ('S003', 24000)], ordered=True)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT staffNo, salary FROM Staff WHERE salary > (SELECT AVG(salary) FROM Staff) ORDER BY salary DESC;
```

- The scalar subquery returns 23000, ignoring Dan’s NULL salary.

</details>

## 13. ANY/SOME, ALL, EXISTS, and correlation (pages 52-56)

- `salary > ANY (subquery)` means greater than at least one returned value; **SOME is a synonym**.
- `salary > ALL (subquery)` means greater than every returned value.
- If returned salaries are `[10000,20000]`, 15000 satisfies >ANY but not >ALL.
- With an empty result, ANY is FALSE and ALL is TRUE. NULL values require three-valued-logic care; replacing these operators by MIN/MAX can change empty/NULL behavior.
- **EXISTS** is TRUE if the subquery returns at least one row; NOT EXISTS is TRUE if none exist.
- EXISTS depends on the existence of rows, not their projected values: a row containing NULL still exists.
- A **correlated subquery** refers to a row of the outer query.

```sql
SELECT staffNo, salary FROM Staff
WHERE salary > ALL (SELECT salary FROM Staff WHERE branchNo = 'B003');

SELECT s.staffNo, s.fName
FROM Staff AS s
WHERE EXISTS (
    SELECT 1 FROM Branch AS b
    WHERE b.branchNo = s.branchNo AND b.city = 'London'
);
```

- In the second query, s is the outer Staff alias; the matching test depends on the current staff member's branchNo.
- Aliases are temporary query names, not new physical tables.
- Think of testing each outer row for explanation; the optimizer need not literally run the subquery from scratch for each row.

### Practice B8

- **Task:** Return clientNo for clients with no viewings.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 1 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([('C003',)], ordered=False)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT c.clientNo FROM Client AS c WHERE NOT EXISTS (SELECT 1 FROM Viewing AS v WHERE v.clientNo = c.clientNo);
```

- The correlated subquery checks whether a viewing exists for each client.

</details>

## 14. Inner joins and multiple tables (pages 57-60)

- Multiple tables without matching conditions produce combinations of rows: n rows and m rows can produce **n*m** combinations.
- A join condition keeps combinations describing the intended relationship.
- Prefer explicit JOIN ... ON for clarity; qualify shared column names using aliases.

```sql
SELECT c.clientNo, c.fName, c.lName, v.propertyNo, v.comment
FROM Client AS c
JOIN Viewing AS v ON c.clientNo = v.clientNo;

SELECT b.branchNo, b.city, s.staffNo, s.fName, p.propertyNo, p.type
FROM Branch AS b
JOIN Staff AS s ON b.branchNo = s.branchNo
JOIN Property AS p ON s.staffNo = p.staffNo
ORDER BY b.branchNo, s.staffNo;
```

- The first join returns one row for each matching viewing; a client who viewed several properties appears several times.
- An inner join excludes clients without matching viewings.
- `JOIN Viewing USING (clientNo)` is concise when both tables have the matching column name. The parentheses omitted on slide 59 are needed.
- The slides vary Client/Clients and Viewing/Viewings; use your actual declared names consistently.

### Practice B9

- **Task:** Exam-style three-table query: return each distinct clientNo, fName, lName for clients who viewed a property with at least 3 rooms.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 2 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([('C001', 'Mia', 'Green'), ('C002', 'Noah', 'Brown')], ordered=False)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT DISTINCT c.clientNo, c.fName, c.lName FROM Client AS c JOIN Viewing AS v ON v.clientNo = c.clientNo JOIN Property AS p ON p.propertyNo = v.propertyNo WHERE p.rooms >= 3;
```

- Client supplies names; Viewing links clients to properties; Property supplies rooms. DISTINCT removes Noah’s repeated qualifying viewings. This practices 2023 Q2(a)’s suggested task, not a full official solution.

</details>

## 15. Outer joins and unmatched records (pages 61-62)

- **INNER JOIN:** only matching pairs.
- **LEFT JOIN:** all left-side rows, with right-side columns NULL when there is no match.
- **RIGHT JOIN:** all right-side rows; equivalent logic can often be written by swapping table order and using LEFT JOIN.
- **FULL OUTER JOIN:** retain unmatched rows on both sides as well as matching pairs; this is general SQL, not native MySQL 8.4 syntax.

```sql
SELECT b.branchNo, b.city, p.propertyNo
FROM Branch AS b
LEFT JOIN Property AS p ON b.city = p.city;
```

- This shows every branch, including a branch with no property in its city.
- Both city columns must actually exist; the slide's bCity/pCity names are placeholders that differ from its earlier schema.
- A right-table predicate in WHERE can remove NULL-extended rows, defeating the intended unmatched-row preservation. Place a match restriction in ON when it should not remove unmatched left rows.
- Supplementary example: to count viewings including clients with none, use LEFT JOIN and COUNT(v.propertyNo), not COUNT(*), which also counts the unmatched placeholder row.

### Practice B10

- **Task:** Return every branchNo and its staff count, including branches with no staff; order by branchNo.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 3 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([('B001', 2), ('B002', 3), ('B003', 0)], ordered=True)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT b.branchNo, COUNT(s.staffNo) FROM Branch AS b LEFT JOIN Staff AS s ON s.branchNo = b.branchNo GROUP BY b.branchNo ORDER BY b.branchNo;
```

- COUNT(s.staffNo) excludes the unmatched NULL placeholder for B003. COUNT(*) would incorrectly count it as one staff member.

</details>

## 16. UNION, INTERSECT, and EXCEPT (pages 63-66)

- These combine **result rows**, whereas joins combine information across columns.
- Operands must have the same number of columns and compatible types at corresponding positions; column names need not match.
- **UNION:** rows in either result. **INTERSECT:** rows in both. **EXCEPT:** rows in the first result but not the second.
- Without ALL, duplicates are removed. ALL preserves the operation's duplicate/multiplicity semantics.

```sql
SELECT city FROM Branch
UNION
SELECT city FROM Property;

SELECT city FROM Branch
INTERSECT
SELECT city FROM Property;

SELECT city FROM Branch
EXCEPT
SELECT city FROM Property;
```

- Example: Branch cities `{London,Glasgow}`, Property cities `{London,Bristol}` give union `{London,Glasgow,Bristol}`, intersection `{London}`, and Branch-minus-Property `{Glasgow}`.
- MySQL 8.4 supports INTERSECT and EXCEPT; older installations can differ. Verify the server version before using them.
- `CORRESPONDING BY` is generic syntax shown in the deck, not MySQL syntax. Select city explicitly instead.
- A DISTINCT/NOT EXISTS query provides an alternative for branch cities without properties; handle NULL city values explicitly if they are allowed.

### Practice B11

- **Task:** Return all distinct city values occurring in Branch or Property, ordered by city.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 3 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
lab.check([('Bristol',), ('Glasgow',), ('London',)], ordered=True)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
SELECT city FROM Branch UNION SELECT city FROM Property ORDER BY city;
```

- UNION combines result rows and removes repeated city values.

</details>

## 17. Advanced constraints: CHECK, domains, assertions (pages 67-69)

- **CHECK** specifies a condition such as `CHECK (salary >= 0)`.
- NULL can yield UNKNOWN, which a CHECK can allow; add NOT NULL if a value is required.
- The slides' sex IN ('M','F') rule is an example domain restriction, not a universal real-world data-design requirement.
- MySQL CHECK constraints do **not** permit subqueries. Use a foreign key to require branchNo to refer to Branch, not the slide's CHECK with SELECT.
- **CREATE DOMAIN:** general SQL concept for a reusable named type with constraints; not supported as that statement in MySQL.
- **CREATE ASSERTION:** general SQL concept for a database-wide condition; it is a constraint, **not a custom data type**. MySQL does not implement that statement.
- Rules unsupported as declarative constraints may need carefully designed transactions, triggers, or application validation; this is an implementation choice beyond the lecture overview.
- MySQL CHECK enforcement has changed across versions. The notes assume a modern version when describing enforcement, not every historical installation.

## 18. Views and indexes (pages 70-72)

- A **view** is a named query exposing a useful subset or presentation of database data.
- It can simplify repeated queries and support restricted access when combined with appropriate permissions.

```sql
CREATE VIEW StaffDirectory AS
SELECT staffNo, fName, lName, branchNo FROM Staff;

SELECT * FROM StaffDirectory;
DROP VIEW StaffDirectory;

CREATE INDEX idx_staff_branch ON Staff (branchNo);
```

- An ordinary view does not automatically store a separate permanent copy of its result; products may execute/merge its query or materialize intermediate results during execution.
- A **materialized view** is a separately maintained stored result, a distinct concept whose support varies by product.
- Not every view is updatable; aggregation and joins can restrict update behavior. WITH CHECK OPTION can constrain changes through eligible filtered views.
- An **index** is an auxiliary structure that can accelerate suitable searches, joins, or ordering.
- The slide's “instantaneous” claim is an oversimplification: indexes do not guarantee constant-time access or eliminate all sorting.
- Indexes cost storage and work during inserts/updates/deletes. Choose them for actual query patterns.
- UNIQUE indexes can enforce uniqueness as well as support lookup.

### Practice B12

- **Task:** Create a view of staffNo and fName for Managers. Then use a separate SQL cell to query it.
- Write your SQL in the next code cell. Run it before the result check.
- Expected output: 2 row(s).

In [ ]:
%%sql
-- Write your answer here.

In [ ]:
%%sql
-- Write the separate SELECT of your view here.

In [ ]:
lab.check([('S001', 'Ada'), ('S003', 'Cara')], ordered=True)

<details>
<summary>Solution and explanation — open after attempting</summary>

```sql
CREATE VIEW IF NOT EXISTS PracticeManagers AS SELECT staffNo, fName FROM Staff WHERE position = 'Manager';

-- Run in a separate %%sql cell:
SELECT staffNo, fName FROM PracticeManagers ORDER BY staffNo;
```

- CREATE VIEW names a query; SELECT reads the view. IF NOT EXISTS makes repeated view creation harmless in this SQLite lab.

</details>

## 19. Transactions and authorization (pages 73-74)

- A **transaction** bundles operations that should succeed or fail together.
- **COMMIT** finalizes changes; **ROLLBACK** discards uncommitted changes within a transaction-capable engine.
- START TRANSACTION begins an explicit transaction in MySQL; autocommit settings affect when individual statements commit.

```sql
START TRANSACTION;
UPDATE Staff SET salary = salary * 1.03 WHERE branchNo = 'B003';
COMMIT;
-- On a failure before commit, use ROLLBACK instead.
```

- ACID, supplementary vocabulary: **atomicity** (all-or-nothing), **consistency** (preserve constraints), **isolation** (control concurrent effects), **durability** (committed data persists).
- Some DDL statements cause implicit commits in MySQL; do not assume every statement can be rolled back like an ordinary InnoDB row update.
- **GRANT** gives privileges; **REVOKE** removes them.
- Example privilege types include SELECT, INSERT, UPDATE, and DELETE.
- The slides show general SQL templates including PUBLIC; MySQL account syntax differs.

```sql
-- Assumes this MySQL account has already been created.
GRANT SELECT ON EstateAgent.Staff TO 'reader'@'localhost';
REVOKE SELECT ON EstateAgent.Staff FROM 'reader'@'localhost';
```

- Database-account permissions and an application's buyer/seller roles are related but different mechanisms; application roles do not automatically become MySQL accounts.

## 20. Summary and mistakes to avoid (page 75)

- **DDL:** define databases, tables, constraints, views, and indexes.
- **DML:** insert, query, update, and delete records.
- WHERE selects rows; HAVING selects groups.
- GROUP BY does not guarantee a sorted result; use ORDER BY.
- COUNT(*) counts rows; COUNT(column) excludes NULL values.
- BETWEEN is inclusive; LIKE is wildcard matching rather than full regex.
- Use IS NULL, not = NULL.
- Scalar subqueries must not return multiple rows; use IN where multiple candidates are intended.
- Include join conditions; understand whether unmatched rows should remain.
- DELETE without WHERE clears records; DROP removes the table.
- Use exact suitable types for money and actual date types for dates.
- Check the schema's names and your server's dialect rather than copying inconsistent slide fragments.

## 21. Revision questions with answers

- **1. Return staff earning at least 20000, highest salary first.**
- **2. Count staff per branch and keep branches with at least three staff.**
- **3. Return staff earning above the overall average.**
- **4. What are COUNT(*) and COUNT(salary) for salaries `[10,NULL,20]`?**
- **5. Why can a LEFT JOIN followed by WHERE on the right table lose unmatched rows?**
- **6. What is the difference between UNION and JOIN?**
- **7. Is DECIMAL(5,2) suitable for salary 21000?**
- **8. Does EXISTS(SELECT NULL ...) return false when a row exists?**

```sql
-- 1
SELECT staffNo, salary FROM Staff
WHERE salary >= 20000 ORDER BY salary DESC;

-- 2
SELECT branchNo, COUNT(*) AS staffCount FROM Staff
GROUP BY branchNo HAVING COUNT(*) >= 3;

-- 3
SELECT staffNo, salary FROM Staff
WHERE salary > (SELECT AVG(salary) FROM Staff);
```

- **4.** 3 and 2; AVG would be 15.
- **5.** Right-side values are NULL for unmatched rows; many comparisons evaluate UNKNOWN, which WHERE excludes.
- **6.** UNION combines compatible result rows; JOIN matches/combines rows from different tables across columns.
- **7.** No: only three integer digits are available, so the positive maximum is 999.99.
- **8.** No. EXISTS checks whether rows exist, regardless of their selected values.

## 22. Official references for MySQL qualifications

- The PDF remains the primary source; these official references support dialect corrections rather than additional lecture requirements.
- [MySQL 8.4 CHECK constraints](https://dev.mysql.com/doc/refman/8.4/en/create-table-check-constraints.html): CHECK restrictions, including no subqueries.
- [MySQL 8.4 foreign-key constraints](https://dev.mysql.com/doc/refman/8.4/en/create-table-foreign-keys.html): supported parent update/delete actions and engine restrictions.
- [MySQL 8.4 SELECT](https://dev.mysql.com/doc/refman/8.4/en/select.html): query syntax and set operations including INTERSECT/EXCEPT.
- [MySQL 8.4 JOIN syntax](https://dev.mysql.com/doc/refman/8.4/en/join.html): supported joins and USING syntax.
- These references describe MySQL 8.4, not a verified installation on your computer.

## Past-paper connection: write SQL that transfers to algebra

- **2023 Q2(a), 2024 Q2(a), 2025 Q2(a):** formulate a business question and query at least three tables (all three in 2024). Later parts reuse your query for algebra, trees, and cost estimates.
- Use consistent aliases, explicit join predicates, and only attributes present in the supplied schema.
- **2025 practice question, different from its example:** Which customers in London have ordered items described as Desk, and what are the order and item IDs?

```sql
SELECT DISTINCT c.custID, c.firstname, c.familyname, o.orderID, i.itemID
FROM Customers AS c
JOIN Orders AS o ON o.custID = c.custID
JOIN LineItems AS li ON li.orderID = o.orderID
JOIN Items AS i ON i.itemID = li.itemID
WHERE c.town = 'London' AND i.description = 'Desk';
```

- All four tables contribute: customer location/names, order ownership, order-item links, and item description.
- DISTINCT makes output set semantics explicit for a later classical relational-algebra translation.
- Explain the query before writing it; check duplicate behavior and whether inner joins match the intended question.
- Later lessons will translate it into algebra/trees and estimate costs. Do not assume those topics are already explained by this SQL notebook.
- Review GROUP BY/HAVING and COUNT/AVG for later distributed aggregates: compute appropriate totals and denominator counts rather than unweighted averages of unequal-site averages.
- See [exam_guide.ipynb](exam_guide.ipynb) for exact references and future-note priorities.

## Practice B13: changing data and rolling back

- Predict Ada’s salary after a 10% increase from 30000. Check the result, then roll back and check again.
- Run these cells in order. The deliberate transaction affects only the disposable lab.
- Expected sequence: 33000 inside the transaction, 30000 after ROLLBACK.
- SQLite transactions here teach the shared concept; MySQL uses START TRANSACTION rather than this BEGIN statement.

In [ ]:
%%sql
BEGIN;

In [ ]:
%%sql
UPDATE Staff SET salary = salary * 1.10 WHERE staffNo = 'S001';

In [ ]:
%%sql
SELECT salary FROM Staff WHERE staffNo = 'S001';

In [ ]:
%%sql
ROLLBACK;

In [ ]:
%%sql
SELECT salary FROM Staff WHERE staffNo = 'S001';

## Restore the lab

- Run the Python cell below whenever you want the original sample records back.
- Reset also removes any views or tables you created in this in-memory lab.

In [ ]:
lab.reset()